In [0]:
# Install all required packages
%pip install clickhouse-connect langchain langchain-community langchain-openai python-dotenv

In [0]:
# Restart Python to use newly installed packages
dbutils.library.restartPython()

In [0]:
import clickhouse_connect

# Connect to ClickHouse using Databricks Secrets
try:
    client = clickhouse_connect.get_client(
        host=dbutils.secrets.get(scope="clickhouse", key="host"),
        port=int(dbutils.secrets.get(scope="clickhouse", key="port")),
        username=dbutils.secrets.get(scope="clickhouse", key="user"),
        password=dbutils.secrets.get(scope="clickhouse", key="password"),
        secure=True
    )
    print(f"✅ Connection successful! ClickHouse version: {client.server_version}")
except Exception as e:
    print(f"❌ Connection failed. Error: {e}")

In [0]:
# Configure LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Academic-ClickHouse-Project"
os.environ["LANGCHAIN_API_KEY"] = os.getenv('LANGCHAIN_API_KEY')

print("📡 LangSmith tracing is now ACTIVE.")

In [0]:
# Create table to store AI logs
client.command("""
CREATE TABLE IF NOT EXISTS ai_logs (
    event_time DateTime DEFAULT now(),
    prompt String,
    response String,
    tokens_used UInt32
) ENGINE = MergeTree()
ORDER BY event_time
""")

print("✅ Table 'ai_logs' is ready in ClickHouse!")

In [0]:
# Adding columns to match your LangGraph AgentState
client.command("ALTER TABLE ai_logs ADD COLUMN IF NOT EXISTS generated_sql String")
client.command("ALTER TABLE ai_logs ADD COLUMN IF NOT EXISTS is_safe UInt8")
client.command("ALTER TABLE ai_logs ADD COLUMN IF NOT EXISTS latency_ms Float64")

print("✅ Columns added: generated_sql, is_safe, latency_ms")

In [0]:
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import PromptTemplate

# Setup Fake LLM with predefined responses
responses = ["The capital of ClickHouse is Speed!", "Data is the new electricity."]
llm = FakeListLLM(responses=responses)

# Create prompt template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Tell me something about {topic}."
)

# Build and run chain using LCEL
topic = "Database Performance"
chain = prompt | llm
ai_response = chain.invoke({"topic": topic})

print(f"🤖 AI says: {ai_response}")

# Save result to ClickHouse
client.command(f"""
INSERT INTO ai_logs (prompt, response, tokens_used) VALUES 
('{topic}', '{ai_response}', 42)
""")

print("✅ Data successfully pushed to ClickHouse & Traced in LangSmith!")

In [0]:
# Query ClickHouse with analytics
query = """
SELECT 
    event_time,
    prompt, 
    response, 
    length(response) AS response_size,
    tokens_used
FROM ai_logs 
ORDER BY event_time DESC 
LIMIT 10
"""

results = client.query(query)

if results.result_rows:
    # Display recent logs
    print(f"{'TIME':<20} | {'PROMPT':<25} | {'SIZE':<6} | {'RESPONSE PREVIEW'}")
    print("-" * 85)
    
    total_chars = 0
    for row in results.result_rows:
        timestamp, prompt, response, size, tokens = row
        preview = response[:35] + "..." if len(response) > 35 else response
        print(f"{str(timestamp):<20} | {prompt:<25} | {size:<6} | {preview}")
        total_chars += size
    
    # Show aggregate statistics
    avg_length = total_chars / len(results.result_rows)
    print(f"\n📊 Statistics: {len(results.result_rows)} records | Avg response: {avg_length:.1f} chars")
else:
    print("No data available in ai_logs table.")